# 05 STATS: daily series and operating modes

Regenerates the daily-series statistics quoted in Section 3.1, the table of onsets and exposure by operating mode (Table 5), and the share of episodes in which the locomotive had moved between onset and recovery (Section 2.5).

Inputs: `DATA/37011FLv4.csv`, `build/fault_episodes_v4.csv`, `build/telemetry_labelled_v4.pkl`; outputs in `OUTPUT/v4/` (the output file names carry an internal suffix, `_v2`).

Paper: Sections 2.5 and 3.1; Table 5.

In [ ]:
import os
import numpy as np, pandas as pd

CFG = dict(
    episodes   = "build/fault_episodes_v4.csv",
    telemetry  = "build/telemetry_labelled_v4.pkl",
    fault_log  = "DATA/37011FLv4.csv",
    outdir     = "OUTPUT/v4",
    series_start = "2024-11-04",
    series_end   = "2025-07-21",
    shed_start   = "2025-05-01",   # annual maintenance, masked as missing
    shed_end     = "2025-05-29",
    june_start = "2025-06-01", june_end = "2025-06-30 23:59:59",
    named_modes = ["POWERING", "PROPULSION", "REGENERATIVE BRAKING", "VCB ON"],
)
os.makedirs(CFG["outdir"], exist_ok=True)

## 1. Daily fault series (Section 3.1)

In [ ]:
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
days = pd.date_range(CFG["series_start"], CFG["series_end"], freq="D")
daily = (ep.set_index("onset").resample("D").size().reindex(days, fill_value=0).astype(float))
shed = (daily.index >= CFG["shed_start"]) & (daily.index <= CFG["shed_end"])
daily[shed] = np.nan
obs = daily.dropna()

stats = {
    "episodes total":            len(ep),
    "calendar days":             len(daily),
    "days masked (shed)":        int(shed.sum()),
    "in-service observations":   len(obs),
    "mean onsets per day":       round(obs.mean(), 2),
    "median onsets per day":     obs.median(),
    "sd":                        round(obs.std(), 2),
    "variance-to-mean ratio":    round(obs.var() / obs.mean(), 2),
    "min":                       obs.min(),
    "max":                       obs.max(),
    "days with no fault":        int((obs == 0).sum()),
    "days below 60":             int((obs < 60).sum()),
    "share below 60 (%)":        round(100 * (obs < 60).mean(), 1),
    "days above 150":            int((obs > 150).sum()),
}
for k, v in stats.items():
    print(f"  {k:28s} {v}")
print("\nTen busiest days")
print(obs.nlargest(10).astype(int).to_string())
print("\nQuantiles")
print(obs.quantile([.1, .25, .5, .75, .9, .95, .99]).round(1).to_string())

daily.rename("onsets").to_csv(f"{CFG['outdir']}/daily_series_v2.csv")
pd.Series(stats).to_csv(f"{CFG['outdir']}/daily_series_stats_v2.csv")

## 2. Table 5: onsets by operating mode

Each June onset is attributed to the operating mode recorded in the telemetry second in which it occurred. Onsets falling in a telemetry gap are reported separately rather than dropped silently. Operating hours are telemetry seconds per mode. Modes outside the four named ones are pooled as "Idle and transitional"; the pool is printed so the grouping is visible.

In [ ]:
tel = pd.read_pickle(CFG["telemetry"])[["DATETIME", "OP"]]
tel = tel[tel.DATETIME.between(CFG["june_start"], CFG["june_end"])]

def group(op):
    return op if op in CFG["named_modes"] else "Idle and transitional"

tel["MODE"] = tel["OP"].map(group)
print("Raw operating modes pooled into 'Idle and transitional':")
print(tel.loc[tel.MODE == "Idle and transitional", "OP"]
         .value_counts().to_string(), "\n")

june = ep[ep.onset.between(CFG["june_start"], CFG["june_end"])].copy()
june = june.merge(tel, left_on="onset", right_on="DATETIME", how="left")
unmatched = int(june.MODE.isna().sum())

t4 = pd.DataFrame({
    "onsets": june.MODE.value_counts(),
    "operating_hours": (tel.MODE.value_counts() / 3600).round(1),
})
t4["onsets_per_hour"] = (t4.onsets / t4.operating_hours).round(2)
t4["share_of_onsets_pct"] = (100 * t4.onsets / t4.onsets.sum()).round(1)
t4["share_of_hours_pct"] = (100 * t4.operating_hours / t4.operating_hours.sum()).round(1)
order = CFG["named_modes"] + ["Idle and transitional"]
t4 = t4.reindex([m for m in order if m in t4.index])

print(t4.to_string())
print(f"\n  June onsets                : {len(june)}")
print(f"  attributed to a mode       : {len(june) - unmatched}")
print(f"  in a telemetry gap         : {unmatched}")
print(f"  telemetry hours in June    : {len(tel) / 3600:.1f}")
t4.to_csv(f"{CFG['outdir']}/table4_operating_mode_v2.csv")

## 3. Fault locations

Locations at onset and at recovery, from the episodes. Runs only if the fault log carries a location column.

In [ ]:
fl = pd.read_csv(CFG["fault_log"])
loc_col = next((c for c in fl.columns if "LOCATION" in c.upper()), None)
print("Fault log columns:", list(fl.columns))

if loc_col is None:
    print("\nNo location column in the fault log; nothing to recompute here.")
else:
    fl["dt"] = pd.to_datetime(fl["DATETIME"], format="%m/%d/%Y %H:%M:%S")
    at = (fl.dropna(subset=[loc_col]).drop_duplicates("dt").set_index("dt")[loc_col])
    m = ep[ep.matched].copy()
    m["loc_onset"] = m.onset.map(at)
    m["loc_recovery"] = m.recovery.map(at)
    both = m.dropna(subset=["loc_onset", "loc_recovery"])
    moved = both.loc_onset != both.loc_recovery
    print(f"\n  distinct locations in the log     : {at.nunique():,}")
    print(f"  matched episodes with both ends   : {len(both):,}")
    print(f"  moved between onset and recovery  : {int(moved.sum()):,} "
          f"({100 * moved.mean():.1f}%)")
    both.assign(moved=moved).to_csv(f"{CFG['outdir']}/episode_locations_v2.csv", index=False)

In [ ]:
print(m.loc_onset.nunique(), m.loc_onset.notna().sum())

___
**END**